# Data Cleaning with Python & Pandas  
# 使用 Python 与 Pandas 进行数据清洗

**Teaching notebook / 教学 Notebook**  
Instructor / 教师: **Waqas Ahmed**

This notebook is designed for undergraduate students who are learning data cleaning for the first time.  
本 Notebook 面向第一次学习数据清洗的本科生。

### Learning objectives / 学习目标
By the end of the class, students should be able to / 完成本节课后，学生应该能够：

1. Read CSV, Excel and JSON data / 读取 CSV、Excel 和 JSON 数据。
2. Inspect a new dataset / 初步检查数据集。
3. Understand common data types / 理解常见数据类型。
4. Find and handle missing values / 发现并处理缺失值。
5. Remove duplicates / 删除重复数据。
6. Clean text and categories / 清理文本和类别。
7. Convert numeric and date columns / 转换数值和日期类型。
8. Detect invalid values and outliers / 检测无效值和异常值。
9. Validate and save the cleaned dataset / 验证并保存清洗后的数据。

## 1. What is data cleaning? / 什么是数据清洗？

**Data cleaning** means finding and correcting problems in raw data before analysis or machine learning.  
**数据清洗**是指在数据分析或机器学习之前，发现并修正原始数据中的问题。

Typical problems / 常见问题：

- Missing values / 缺失值
- Wrong data types / 错误的数据类型
- Duplicate rows / 重复行
- Extra spaces or inconsistent text / 多余空格或文本不一致
- Invalid values / 无效数值
- Inconsistent categories / 类别不一致
- Wrong date formats / 日期格式错误
- Outliers / 异常值

A useful rule / 一个重要原则：

> **Never clean blindly. First inspect, then decide.**  
> **不要盲目清洗。先检查，再决定。**

## 2. Import the libraries / 导入库

We will mainly use **pandas** for tabular data and **NumPy** for missing values and numerical operations.  
我们主要使用 **pandas** 处理表格数据，使用 **NumPy** 处理缺失值和数值运算。

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

print('pandas version:', pd.__version__)

## 3. Locate the teaching dataset / 找到教学数据集

The supplied file is `student_data_messy.csv`. It intentionally contains mistakes so we can clean them in class.  
提供的 `student_data_messy.csv` 故意包含一些错误，便于课堂演示清洗过程。

In [ ]:
from pathlib import Path

DATA_FILE = Path('student_data_messy.csv')

# When running from another folder, also try the course folder.
if not DATA_FILE.exists():
    candidate = Path('/mnt/data/data_cleaning_course/student_data_messy.csv')
    if candidate.exists():
        DATA_FILE = candidate

print('Using file:', DATA_FILE.resolve())

# Part A — Reading Data / 第一部分：读取数据

## 4. Read a CSV file / 读取 CSV 文件

`pd.read_csv()` is one of the most common pandas functions.  
`pd.read_csv()` 是 pandas 中最常用的函数之一。

```python
df = pd.read_csv('file.csv')
```

Important options / 常用参数：

- `sep=','` : separator / 分隔符
- `header=0` : row containing column names / 列名所在行
- `names=[...]` : provide your own column names / 自定义列名
- `usecols=[...]` : read selected columns / 只读取指定列
- `nrows=100` : read first 100 rows / 只读前 100 行
- `encoding='utf-8'` : text encoding / 文本编码
- `na_values=['NA','?','missing']` : symbols treated as missing / 指定缺失值符号

In [ ]:
df = pd.read_csv(DATA_FILE)
print('Data loaded successfully! / 数据读取成功！')
df.head()

## 5. Read Excel / 读取 Excel

For Excel files we use `pd.read_excel()`.  
读取 Excel 文件使用 `pd.read_excel()`。

```python
df = pd.read_excel('students.xlsx', sheet_name='Sheet1')
```

Useful options / 常用参数：
- `sheet_name=0` or `'Sheet1'` / 工作表名称
- `usecols='A:F'` / 指定列
- `skiprows=2` / 跳过前两行

Below we create an Excel copy only for demonstration. / 下面先创建一个 Excel 副本用于演示。

In [ ]:
excel_file = Path('student_data_messy.xlsx')
df.to_excel(excel_file, index=False)

df_excel = pd.read_excel(excel_file)
df_excel.head(3)

## 6. Read JSON / 读取 JSON

JSON is common in web APIs and software systems. / JSON 常见于 Web API 和软件系统。

```python
df = pd.read_json('data.json')
```

For table-like JSON, pandas can often read it directly. / 对于表格形式的 JSON，pandas 通常可以直接读取。

In [ ]:
json_file = Path('student_data_messy.json')
df.to_json(json_file, orient='records', indent=2, force_ascii=False)

df_json = pd.read_json(json_file)
df_json.head(3)

# Part B — First Inspection / 第二部分：初步检查数据

## 7. First five rows, last rows, random rows / 查看前几行、后几行、随机行

Use these before changing anything. / 在修改数据之前先使用这些函数。

In [ ]:
print('HEAD / 前5行')
display(df.head())

print('TAIL / 后5行')
display(df.tail())

print('SAMPLE / 随机3行')
display(df.sample(3, random_state=42))

## 8. Shape, columns and index / 数据形状、列名和索引

- `df.shape` → `(number_of_rows, number_of_columns)` / 行数和列数
- `df.columns` → column names / 列名
- `df.index` → row labels / 行索引

In [ ]:
print('Shape / 形状:', df.shape)
print('\nColumns / 列名:')
print(df.columns.tolist())
print('\nIndex / 索引:')
print(df.index)

## 9. `info()` — one of the most important commands / 最重要的检查命令之一

`df.info()` shows / `df.info()` 可以显示：

- column names / 列名
- non-null counts / 非空值数量
- pandas data types / 数据类型
- approximate memory use / 大致内存使用

In [ ]:
df.info()

## 10. `describe()` — numerical summary / 数值统计摘要

For numeric columns it shows / 对数值列可以显示：

- `count` number of non-missing values / 非缺失数量
- `mean` average / 平均值
- `std` standard deviation / 标准差
- `min`, `max` / 最小值和最大值
- `25%`, `50%`, `75%` quartiles / 四分位数

Because several columns are currently stored as text, the first result may be limited. This itself is a clue that cleaning is required.  
由于某些本应是数值的列目前被读成文本，统计结果会受限制——这本身就是需要清洗的信号。

In [ ]:
df.describe()

In [ ]:
# Include text/object columns too / 也包含文本列

df.describe(include='all').T

# Part C — Data Types in Detail / 第三部分：详细理解数据类型

## 11. Why data types matter / 为什么数据类型很重要？

A computer must know **how to interpret each value**. / 计算机必须知道**如何解释每个值**。

Examples / 示例：

| Meaning / 含义 | Example / 示例 | Typical pandas dtype / 常见类型 |
|---|---:|---|
| Whole number / 整数 | `21` | `int64`, `Int64` |
| Decimal number / 小数 | `92.5` | `float64` |
| Text / 文本 | `'Wuhan'` | `object` or `string` |
| True/False / 真/假 | `True` | `bool` |
| Category / 类别 | `'Male'`, `'Female'` | `category` |
| Date & time / 日期时间 | `2026-02-15` | `datetime64[ns]` |

**Important:** A column containing mostly numbers can still become `object` if one cell contains text such as `'twenty'`.  
**重要：** 即使一列大多数都是数字，只要其中一个单元格含有 `'twenty'` 这样的文本，整列就可能被读取为 `object`。

## 12. Integer types / 整数类型

Examples: age, count, ID. / 例如：年龄、数量、编号。

- `int64` cannot contain standard `NaN`. / `int64` 通常不能直接存储 `NaN`。
- pandas nullable integer `Int64` **can** contain `<NA>`. / pandas 的可空整数 `Int64` 可以保存 `<NA>`。

Student IDs are usually **identifiers**, not quantities. We normally do not calculate their average.  
学生编号虽然由数字组成，但它们通常是**标识符**，而不是可用于计算平均值的数量。

In [ ]:
print(df['Student_ID'].dtype)
print(df['Student_ID'].head())

## 13. Floating-point types / 浮点数类型

Examples: height, weight, temperature, probability. / 例如：身高、体重、温度、概率。

`float64` can store decimal values and `NaN`. / `float64` 可以存储小数和 `NaN`。

Be careful with exact equality for floating-point values. / 浮点数进行精确相等比较时需要小心。

## 14. String / object types / 字符串与 object 类型

Text columns often arrive as `object`. Newer pandas also supports `string`.  
文本列经常被读取为 `object`，新版 pandas 也支持 `string`。

Typical cleaning operations / 常见清洗操作：

- remove spaces / 去除空格
- change upper/lower case / 统一大小写
- replace spelling variants / 统一不同拼写
- extract parts using string methods / 使用字符串函数提取内容

In [ ]:
print(df['Name'].dtype)
print(df['Name'].head().tolist())

## 15. Boolean type / 布尔类型

Boolean data has two logical values: `True` and `False`. / 布尔数据只有两个逻辑值：`True` 和 `False`。

Examples / 示例：
- `Passed = True/False`
- `Is_Missing = True/False`
- `Attendance_pct >= 75`

Boolean masks are fundamental for filtering rows. / 布尔掩码是数据筛选的基础。

In [ ]:
# We will create a temporary boolean condition.
# 先创建一个临时布尔条件。

pd.to_numeric(df['Attendance_pct'], errors='coerce').ge(75).head(10)

## 16. Category type / 类别类型

A category column contains values from a limited set. / 类别列只包含有限的几个取值。

Examples / 示例：
- Gender / 性别
- City / 城市
- Major / 专业

Benefits / 优点：
- can use less memory / 可以减少内存
- makes the meaning clearer / 语义更明确
- useful for machine-learning preprocessing / 便于后续机器学习预处理

## 17. Datetime type / 日期时间类型

Dates should usually be converted from text to pandas datetime. / 日期通常应该从文本转换成 pandas 日期类型。

After conversion we can easily obtain / 转换后可以方便得到：
- year / 年
- month / 月
- day / 日
- weekday / 星期
- time differences / 时间差

In [ ]:
# Example only; we will clean the whole column later.
example_dates = pd.to_datetime(pd.Series(['2026-02-15', '2026/02/16']), errors='coerce', format='mixed')
print(example_dates)
print('Year:', example_dates.dt.year.tolist())

## 18. Converting data types safely / 安全地转换数据类型

Common tools / 常用工具：

```python
pd.to_numeric(series, errors='coerce')
pd.to_datetime(series, errors='coerce')
series.astype('string')
series.astype('category')
```

`errors='coerce'` changes invalid values into missing values instead of crashing the program.  
`errors='coerce'` 会把无法转换的值变成缺失值，而不是让程序报错停止。

# Part D — Cleaning the Dataset / 第四部分：开始清洗数据

## 19. Always work on a copy / 始终在副本上操作

During teaching and exploration, keep the raw dataset unchanged. / 教学和探索时，最好保留原始数据不变。

In [ ]:
clean = df.copy()
print('Raw shape / 原始形状:', df.shape)
print('Working copy / 工作副本:', clean.shape)

## 20. Clean column names / 清理列名

Good column names are consistent and easy to type. / 好的列名应该统一、容易输入。

Typical rules / 常见规则：
- remove leading/trailing spaces / 去掉首尾空格
- use lower case / 使用小写
- replace spaces with `_` / 空格替换为下划线

In [ ]:
clean.columns = (
    clean.columns
         .str.strip()
         .str.lower()
         .str.replace(' ', '_', regex=False)
)

clean.columns.tolist()

## 21. Detect missing values / 检测缺失值

Useful commands / 常用命令：

```python
df.isna()
df.isna().sum()
df.isna().mean() * 100
```

But remember: values like `''`, `'NA'`, `'N/A'`, `'?'`, `'missing'` may not automatically be recognized in every situation.  
注意：空字符串、`NA`、`N/A`、`?`、`missing` 等有时不会自动被识别为缺失值。

In [ ]:
missing_table = pd.DataFrame({
    'missing_count': clean.isna().sum(),
    'missing_percent': clean.isna().mean().mul(100).round(1)
})
missing_table.sort_values('missing_count', ascending=False)

## 22. Replace missing-value markers / 统一缺失值符号

We can convert common text markers into `NaN`. / 可以把常见的文本缺失符号统一转换为 `NaN`。

In [ ]:
missing_markers = ['', ' ', 'NA', 'N/A', 'na', 'n/a', '?', 'missing', 'Missing']
clean = clean.replace(missing_markers, np.nan)

clean.isna().sum().sort_values(ascending=False)

## 23. Remove extra spaces in text / 去除文本中的多余空格

`str.strip()` removes spaces at the beginning and end. / `str.strip()` 去掉字符串首尾的空格。

In [ ]:
text_cols = clean.select_dtypes(include=['object', 'string']).columns

for col in text_cols:
    clean[col] = clean[col].astype('string').str.strip()

clean[['name', 'gender', 'major', 'email', 'city']].head(8)

## 24. Inspect unique categories / 检查类别的不同写法

Before replacing categories, first see what values actually exist. / 在统一类别之前，先查看当前有哪些不同取值。

In [ ]:
for col in ['gender', 'major', 'city']:
    print(f'--- {col} ---')
    print(clean[col].value_counts(dropna=False))
    print()

## 25. Standardize gender / 统一性别类别

The same meaning may appear as `M`, `male`, `MALE`, `F`, `female`, etc.  
同一个含义可能写成 `M`、`male`、`MALE`、`F`、`female` 等不同形式。

A mapping dictionary is a clear way to standardize them. / 使用映射字典是一种清晰的统一方式。

In [ ]:
gender_map = {
    'm': 'Male',
    'male': 'Male',
    'f': 'Female',
    'female': 'Female'
}

clean['gender'] = clean['gender'].str.lower().map(gender_map)
clean['gender'].value_counts(dropna=False)

## 26. Standardize major / 统一专业名称

We will map abbreviations and spelling variants to three standard names. / 将缩写和不同写法统一为三个标准专业名称。

In [ ]:
major_key = clean['major'].str.lower().str.replace('.', '', regex=False)

major_map = {
    'cs': 'Computer Science',
    'computer science': 'Computer Science',
    'ai': 'Artificial Intelligence',
    'artificial intelligence': 'Artificial Intelligence',
    'data science': 'Data Science'
}

clean['major'] = major_key.map(major_map)
clean['major'].value_counts(dropna=False)

## 27. Standardize city capitalization / 统一城市名称大小写

`str.title()` makes the first letter of each word uppercase. / `str.title()` 将每个单词首字母转换为大写。

In [ ]:
clean['city'] = clean['city'].str.title()
clean['city'].value_counts(dropna=False)

## 28. Convert numeric columns / 转换数值列

Several columns should be numeric but contain invalid text. / 有些列本应为数值，但包含非法文本。

`pd.to_numeric(..., errors='coerce')` converts valid numbers and changes invalid entries to `NaN`.  
`pd.to_numeric(..., errors='coerce')` 会转换合法数字，并将非法值变成 `NaN`。

In [ ]:
numeric_cols = [
    'age', 'attendance_pct', 'homework', 'presentation',
    'project', 'height_cm', 'weight_kg'
]

for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')

clean[numeric_cols].dtypes

## 29. Validate ranges / 检查数值范围

A value can be numeric but still be **impossible** or **invalid**. / 一个值即使是数字，也可能不合理或无效。

Example rules for this classroom dataset / 本课堂示例的规则：

- Age: 16–80 / 年龄 16–80
- Attendance: 0–100% / 出勤率 0–100%
- Homework: 0–20 / 作业 0–20
- Presentation: 0–20 / 展示 0–20
- Project: 0–50 / 项目 0–50
- Height: 130–220 cm / 身高 130–220 cm
- Weight: 35–200 kg / 体重 35–200 kg

These ranges come from **domain knowledge**. / 这些范围来自**领域知识**。

In [ ]:
rules = {
    'age': (16, 80),
    'attendance_pct': (0, 100),
    'homework': (0, 20),
    'presentation': (0, 20),
    'project': (0, 50),
    'height_cm': (130, 220),
    'weight_kg': (35, 200),
}

for col, (low, high) in rules.items():
    invalid = ~clean[col].between(low, high) & clean[col].notna()
    if invalid.any():
        print(f'{col}: invalid values / 无效值 ->', clean.loc[invalid, col].tolist())

## 30. Replace impossible values with missing / 将不合理值设为缺失

We should not invent corrections without evidence. / 没有证据时，不应该随意猜测正确值。

A safe teaching strategy is to mark impossible values as missing, then decide how to handle them.  
一种安全的做法是先将不合理值标记为缺失，再决定后续处理方法。

In [ ]:
for col, (low, high) in rules.items():
    clean.loc[~clean[col].between(low, high), col] = np.nan

clean[numeric_cols].isna().sum()

## 31. Parse dates / 解析日期

Real data may use several date formats. / 真实数据可能包含多种日期格式。

`errors='coerce'` turns invalid dates into `NaT` (Not a Time). / 无法解析的日期会变成 `NaT`。

In [ ]:
clean['enrollment_date'] = pd.to_datetime(
    clean['enrollment_date'],
    errors='coerce',
    format='mixed',
    dayfirst=False
)

clean[['enrollment_date']].head(12)

In [ ]:
print('Invalid/missing dates / 无效或缺失日期:')
display(clean.loc[clean['enrollment_date'].isna(), ['student_id', 'name', 'enrollment_date']])

## 32. Find duplicate rows / 查找重复行

Duplicates can happen when data is entered twice or files are combined incorrectly. / 重复数据可能来自重复录入或错误合并。

```python
df.duplicated()
df.duplicated().sum()
df.drop_duplicates()
```

In [ ]:
print('Number of duplicate rows / 重复行数量:', clean.duplicated().sum())

display(clean.loc[clean.duplicated(keep=False)])

In [ ]:
clean = clean.drop_duplicates().reset_index(drop=True)
print('Shape after removing duplicates / 删除重复后:', clean.shape)

## 33. Missing values: drop or fill? / 缺失值：删除还是填补？

There is no single correct answer. / 没有唯一正确答案。

### Option A: Drop / 删除
Useful when / 适合：
- only a few rows are missing / 缺失行很少
- the row is unusable / 该行无法使用

### Option B: Fill / 填补
Useful when / 适合：
- losing rows is costly / 删除会损失太多数据
- a reasonable replacement exists / 有合理的替代值

Common choices / 常见方法：
- mean / 均值
- median / 中位数
- mode / 众数
- group-based value / 按组填补
- interpolation / 插值

**Never fill without thinking about the meaning of the variable.**  
**不要在不了解变量含义的情况下盲目填补。**

## 34. Fill numerical missing values with the median / 用中位数填补数值缺失

Median is often safer than mean when outliers may exist. / 当存在异常值时，中位数通常比均值更稳健。

In [ ]:
fill_numeric = ['age', 'attendance_pct', 'homework', 'presentation', 'project', 'height_cm', 'weight_kg']

for col in fill_numeric:
    median_value = clean[col].median()
    clean[col] = clean[col].fillna(median_value)
    print(f'{col}: median / 中位数 = {median_value}')

## 35. Fill categorical missing values / 填补类别缺失值

For a classroom example, we can fill a missing category with the mode (most frequent value).  
在课堂示例中，可以使用众数（出现次数最多的类别）进行填补。

In real research, sometimes it is better to keep a separate `'Unknown'` category. / 在真实研究中，有时保留 `'Unknown'` 类别更合理。

In [ ]:
for col in ['gender', 'major', 'city']:
    mode_value = clean[col].mode(dropna=True)[0]
    clean[col] = clean[col].fillna(mode_value)
    print(f'{col}: mode / 众数 = {mode_value}')

## 36. Handling a missing or invalid date / 处理缺失或无效日期

For dates, blindly filling with an average date is often meaningless. / 对日期直接使用平均值填补通常没有意义。

For this teaching example we will leave invalid dates as `NaT` so students can clearly see that some information is unknown.  
本示例中保留 `NaT`，表示该日期信息未知。

## 37. Outliers / 异常值

An outlier is a value far from most observations. / 异常值是与大多数观测值相距较远的数值。

**Important:** An outlier is not automatically an error. / **重要：异常值不一定是错误。**

It could be / 它可能是：
- a data-entry mistake / 录入错误
- a measurement error / 测量误差
- a rare but real case / 少见但真实的情况

Always investigate before deleting. / 删除前必须先调查。

## 38. IQR method / IQR 四分位距方法

For a numeric feature / 对数值特征：

\[
IQR = Q_3 - Q_1
\]

A common rule marks values outside / 常用规则将以下范围外的数据视为潜在异常值：

\[
[Q_1 - 1.5\,IQR,\; Q_3 + 1.5\,IQR]
\]

This is a **screening rule**, not an automatic deletion rule. / 这是**筛查规则**，不是自动删除规则。

In [ ]:
def iqr_outlier_mask(series, factor=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

mask, lower, upper = iqr_outlier_mask(clean['weight_kg'])
print('Weight IQR limits / 体重 IQR 范围:', round(lower, 2), 'to', round(upper, 2))
display(clean.loc[mask, ['student_id', 'name', 'weight_kg']])

## 39. Inspect distributions / 检查数据分布

A box plot helps us visually inspect spread and potential outliers. / 箱线图可以帮助我们直观检查数据分布和潜在异常值。

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.boxplot(clean['weight_kg'].dropna(), vert=False)
plt.xlabel('Weight (kg)')
plt.title('Weight Distribution / 体重分布')
plt.show()

## 40. Clean email text / 清理邮箱文本

Text fields may contain hidden spaces. / 文本字段可能包含隐藏空格。

We can also perform simple checks, for example whether an email contains `@`. / 还可以做简单检查，例如邮箱是否包含 `@`。

In [ ]:
clean['email'] = clean['email'].str.strip().str.lower()
clean['email_valid'] = clean['email'].str.contains('@', na=False)

clean[['email', 'email_valid']].head(10)

## 41. Convert stable categories to `category` dtype / 将稳定类别转换为 category 类型

In [ ]:
for col in ['gender', 'major', 'city']:
    clean[col] = clean[col].astype('category')

clean.dtypes

## 42. Convert age to nullable integer / 将年龄转换为可空整数

Age is conceptually a whole number, so after cleaning we can store it as pandas `Int64`.  
年龄在概念上是整数，因此清洗后可以使用 pandas 的 `Int64`。

In [ ]:
clean['age'] = clean['age'].round().astype('Int64')
clean['age'].dtype

# Part E — Useful Inspection and Filtering / 第五部分：常用检查与筛选

## 43. `value_counts()` and `unique()` / 类别计数与唯一值

These are extremely useful for spotting inconsistent categories. / 这两个函数对于发现类别不一致非常有用。

In [ ]:
print(clean['major'].value_counts())
print('\nUnique cities / 不同城市:', clean['city'].unique().tolist())

## 44. Filter rows using conditions / 使用条件筛选行

Example: students with attendance below 75%. / 示例：筛选出勤率低于 75% 的学生。

In [ ]:
low_attendance = clean.loc[
    clean['attendance_pct'] < 75,
    ['student_id', 'name', 'attendance_pct']
]
low_attendance

## 45. `loc`, `iloc`, and `query` / 三种常用选择方法

- `loc` → labels and conditions / 按标签和条件选择
- `iloc` → integer positions / 按位置选择
- `query` → readable expression / 使用可读表达式筛选

In [ ]:
print('loc example / loc 示例')
display(clean.loc[0:3, ['name', 'major']])

print('iloc example / iloc 示例')
display(clean.iloc[:3, :5])

print('query example / query 示例')
display(clean.query('attendance_pct >= 90')[['name', 'attendance_pct']].head())

# Part F — Validation / 第六部分：清洗后的验证

## 46. Check missing values again / 再次检查缺失值

In [ ]:
clean.isna().sum().sort_values(ascending=False)

## 47. Check duplicates again / 再次检查重复值

In [ ]:
print('Duplicates / 重复行:', clean.duplicated().sum())
print('Duplicate Student_ID / 重复学号:', clean['student_id'].duplicated().sum())

## 48. Re-check ranges / 再次验证数值范围

In [ ]:
checks = {
    'age_valid': clean['age'].between(16, 80).all(),
    'attendance_valid': clean['attendance_pct'].between(0, 100).all(),
    'homework_valid': clean['homework'].between(0, 20).all(),
    'presentation_valid': clean['presentation'].between(0, 20).all(),
    'project_valid': clean['project'].between(0, 50).all(),
    'height_valid': clean['height_cm'].between(130, 220).all(),
    'weight_valid': clean['weight_kg'].between(35, 200).all(),
}
checks

## 49. View the cleaned data / 查看清洗后的数据

In [ ]:
clean.head(10)

## 50. Compare before and after / 清洗前后比较

In [ ]:
summary = pd.DataFrame({
    'raw': [len(df), df.duplicated().sum(), int(df.isna().sum().sum())],
    'cleaned': [len(clean), clean.duplicated().sum(), int(clean.isna().sum().sum())]
}, index=['rows / 行数', 'duplicate rows / 重复行', 'missing cells / 缺失单元格'])
summary

## 51. Save the cleaned dataset / 保存清洗后的数据

Always save cleaned data to a **new file**. Do not overwrite the raw source unless you are absolutely certain.  
清洗后的数据应保存为**新文件**，不要轻易覆盖原始文件。

In [ ]:
clean_csv = Path('student_data_cleaned.csv')
clean_excel = Path('student_data_cleaned.xlsx')

clean.to_csv(clean_csv, index=False)
clean.to_excel(clean_excel, index=False)

print('Saved / 已保存:', clean_csv.resolve())
print('Saved / 已保存:', clean_excel.resolve())

# Part G — A Reusable Cleaning Function / 第七部分：可重复使用的清洗函数

In real projects, we do not want to manually repeat the same cleaning commands every time.  
在真实项目中，我们不希望每次都手动重复相同的清洗命令。

A function makes the process reproducible. / 使用函数可以让清洗流程可重复。

In [ ]:
def clean_student_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    data = raw_df.copy()

    # 1. Column names / 列名
    data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_', regex=False)

    # 2. Missing markers / 缺失值符号
    data = data.replace(['', ' ', 'NA', 'N/A', 'na', 'n/a', '?', 'missing'], np.nan)

    # 3. Trim text / 去空格
    text_cols = data.select_dtypes(include=['object', 'string']).columns
    for col in text_cols:
        data[col] = data[col].astype('string').str.strip()

    # 4. Standardize categories / 统一类别
    gender_map = {'m': 'Male', 'male': 'Male', 'f': 'Female', 'female': 'Female'}
    data['gender'] = data['gender'].str.lower().map(gender_map)

    major_key = data['major'].str.lower().str.replace('.', '', regex=False)
    major_map = {
        'cs': 'Computer Science', 'computer science': 'Computer Science',
        'ai': 'Artificial Intelligence', 'artificial intelligence': 'Artificial Intelligence',
        'data science': 'Data Science'
    }
    data['major'] = major_key.map(major_map)
    data['city'] = data['city'].str.title()

    # 5. Numeric conversion / 数值转换
    numeric_cols = ['age', 'attendance_pct', 'homework', 'presentation', 'project', 'height_cm', 'weight_kg']
    for col in numeric_cols:
        data[col] = pd.to_numeric(data[col], errors='coerce')

    # 6. Range rules / 合理范围
    ranges = {
        'age': (16, 80), 'attendance_pct': (0, 100),
        'homework': (0, 20), 'presentation': (0, 20),
        'project': (0, 50), 'height_cm': (130, 220), 'weight_kg': (35, 200)
    }
    for col, (low, high) in ranges.items():
        data.loc[~data[col].between(low, high), col] = np.nan

    # 7. Dates / 日期
    data['enrollment_date'] = pd.to_datetime(data['enrollment_date'], errors='coerce', format='mixed')

    # 8. Duplicates / 重复行
    data = data.drop_duplicates().reset_index(drop=True)

    # 9. Fill selected missing values / 填补部分缺失值
    for col in numeric_cols:
        data[col] = data[col].fillna(data[col].median())
    for col in ['gender', 'major', 'city']:
        data[col] = data[col].fillna(data[col].mode(dropna=True)[0])

    # 10. Final dtypes / 最终类型
    data['age'] = data['age'].round().astype('Int64')
    for col in ['gender', 'major', 'city']:
        data[col] = data[col].astype('category')

    data['email'] = data['email'].str.strip().str.lower()
    data['email_valid'] = data['email'].str.contains('@', na=False)

    return data

# Test the function / 测试函数
clean2 = clean_student_data(df)
clean2.head()

# Part H — Classroom Exercises / 第八部分：课堂练习

Try each task before looking at the solution. / 请先自己完成，再查看答案。

### Exercise 1 / 练习 1
Read the CSV again and display only the first 3 rows. / 重新读取 CSV，并显示前 3 行。

In [ ]:
# Write your answer here / 在这里写答案


### Exercise 2 / 练习 2
Find the number of missing values in every column. / 统计每一列的缺失值数量。

In [ ]:
# Write your answer here / 在这里写答案


### Exercise 3 / 练习 3
Show every unique value in the original `Gender` column. / 显示原始 `Gender` 列中的所有唯一值。

In [ ]:
# Write your answer here / 在这里写答案


### Exercise 4 / 练习 4
Convert the original `Age` column to numeric using `errors='coerce'`. Which values become missing?  
使用 `errors='coerce'` 将原始 `Age` 列转换为数值。哪些值变成了缺失值？

In [ ]:
# Write your answer here / 在这里写答案


### Exercise 5 / 练习 5
Find students whose cleaned attendance is below 80%. / 找出清洗后出勤率低于 80% 的学生。

In [ ]:
# Write your answer here / 在这里写答案


## Exercise solutions / 练习答案

In [ ]:
# Solution 1 / 答案 1
pd.read_csv(DATA_FILE).head(3)

In [ ]:
# Solution 2 / 答案 2
pd.read_csv(DATA_FILE).isna().sum()

In [ ]:
# Solution 3 / 答案 3
pd.read_csv(DATA_FILE)['Gender'].unique()

In [ ]:
# Solution 4 / 答案 4
age_numeric = pd.to_numeric(pd.read_csv(DATA_FILE)['Age'], errors='coerce')
pd.DataFrame({'original': pd.read_csv(DATA_FILE)['Age'], 'numeric': age_numeric}).loc[age_numeric.isna()]

In [ ]:
# Solution 5 / 答案 5
clean.loc[clean['attendance_pct'] < 80, ['student_id', 'name', 'attendance_pct']]

# Final Checklist / 最终检查清单

Before using a dataset for machine learning, ask / 在将数据用于机器学习之前，请检查：

- [ ] Did I inspect the raw data? / 是否检查了原始数据？
- [ ] Are column names clean? / 列名是否规范？
- [ ] Are data types correct? / 数据类型是否正确？
- [ ] Did I check missing values? / 是否检查缺失值？
- [ ] Did I check duplicates? / 是否检查重复值？
- [ ] Are categories consistent? / 类别是否统一？
- [ ] Are numeric ranges reasonable? / 数值范围是否合理？
- [ ] Are dates parsed correctly? / 日期是否正确解析？
- [ ] Did I investigate outliers? / 是否检查异常值？
- [ ] Did I validate after cleaning? / 清洗后是否重新验证？
- [ ] Did I save a new cleaned file? / 是否保存了新的清洗数据文件？

## Key message / 核心结论

> **Good machine learning begins with good data.**  
> **高质量的机器学习始于高质量的数据。**